# 🔢 텍스트 임베딩 이해하기

임베딩의 세계에 오신 것을 환영합니다! RAG를 자세히 살펴보기 전에, RAG를 가능하게 하는 기본 개념인 **텍스트 임베딩** 🪄을 이해해 보겠습니다.

임베딩은 단어, 문장, 심지어 하나의 섹션 전체를 숫자로 변환하여 컴퓨터에게 텍스트의 *의미*를 가르치는 방법이라고 생각하면 됩니다. 이 단순한 개념이 AI 시스템이 텍스트를 이해하고 검색하는 방식을 혁신적으로 바꿉니다.

이 마법이 어떻게 작동하는지 살펴보겠습니다! ✨

## 📚 라이브러리 가져오기

임베딩을 만들고 유사도를 측정하는 데 사용할 도구를 가져옵니다.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("🧠 Ready to explore embeddings!")

## 🔢 단어에서 숫자로: 간단한 예시

서로 다른 단어가 어떻게 서로 다른 벡터가 되는지 보여 줍니다(설명을 위해 4차원으로 단순화했습니다).

- `"dog"` → [0.2, 0.8, 0.1, 0.6] 
- `"puppy"` → [0.3, 0.7, 0.2, 0.5] (dog와 비슷함 - 둘 다 개과)
- `"cat"` → [0.1, 0.6, 0.8, 0.3] (다르지만 어느 정도 비슷함 - 둘 다 반려동물)
- `"car"` → [0.9, 0.1, 0.2, 0.1] (완전히 다름 - 동물이 아님)

실제 임베딩에서는 이 벡터가 수백에서 수천 차원을 가지며, 미묘한 의미와 의미론적 정보를 담아냅니다!

## 🧪 실제 임베딩 만들어 보기!

이제 실제로 확인해 보겠습니다! 앞의 단어들을 실제 임베딩 벡터로 변환해 봅니다.  
이를 위해 **임베딩 모델**을 사용합니다. 임베딩 모델은 입력된 단어와 문장에 대한 LLM의 "내부 표현"을 꺼내 쓰는 LLM입니다. 이 출력(임베딩)은 보통 벡터로 표현되며, 학습된 LLM 내부에서 나온 값이므로 의미론적 의미를 갖습니다.

In [ ]:
# 사전 학습된 임베딩 모델 불러오기
model = SentenceTransformer('all-MiniLM-L6-v2')

# 위 예시에서 사용한 단어들
words = ["dog", "puppy", "cat", "car"]

# 단어를 임베딩(벡터)으로 변환
embeddings = model.encode(words)

print("🔢 단어를 벡터로 변환:")
for word, embedding in zip(words, embeddings):
    print(f"'{word}' → {len(embedding)}차원 벡터")
    print(f"   처음 5개 값: {embedding[:5]}")
    print()

## ✨ 벡터의 마법

1. **텍스트 → 숫자**: 모든 텍스트는 그 의미를 담은 숫자 목록(벡터)으로 변환됩니다
2. **유사도 매칭**: 질문을 하면 시스템이 거리가 가까운 벡터를 찾습니다
3. **매우 빠른 성능**: 문서가 수백만 개라도 검색은 밀리초 단위로 이루어집니다

## 🔍 유사도 테스트하기

단어들이 서로 얼마나 비슷한지 측정해 보겠습니다. 어떤 단어끼리 가장 관련이 깊은지 컴퓨터가 알려 줍니다!

In [ ]:
# 'dog'와 다른 모든 단어 사이의 유사도 계산
dog_embedding = embeddings[0].reshape(1, -1)  # 'dog'가 첫 번째 단어
other_embeddings = embeddings[1:]  # puppy, cat, car

# 코사인 유사도 계산(높을수록 더 비슷함)
similarities = cosine_similarity(dog_embedding, other_embeddings)[0]

print("🐕 'dog'는 다른 단어들과 얼마나 유사할까요?")
print()
for word, similarity in zip(words[1:], similarities):
    print(f"dog ↔ {word}: {similarity:.3f}")

print()
print("📊 살펴볼 점:")
print("• 'dog'와 'puppy'는 가장 유사합니다 (둘 다 개과 동물)")
print("• 'dog'와 'cat'은 어느 정도 유사합니다 (둘 다 반려동물)")
print("• 'dog'와 'car'는 유사도가 가장 낮습니다 (완전히 다른 개념)")

## 🗺️ 의미의 지도

단어를 공간 위의 점으로 그려 볼 수 있습니다. 의미가 비슷한 단어는 서로 가까이 모이고, 의미가 다른 단어는 멀리 떨어집니다.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def plot_embedding_map(labels, embeddings, title='Words plotted by meaning — closer = more similar'):
    coords = PCA(n_components=2).fit_transform(embeddings)
    x_pad = (coords[:, 0].max() - coords[:, 0].min()) * 0.5
    y_pad = (coords[:, 1].max() - coords[:, 1].min()) * 0.5

    palette = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71', '#9b59b6', '#1abc9c', '#e91e63', '#ff9800']
    colors = palette[:len(labels)]

    fig, ax = plt.subplots(figsize=(7, 6))
    fig.patch.set_facecolor('#f9f7ff')
    ax.set_facecolor('#f9f7ff')

    for label, (x, y), color in zip(labels, coords, colors):
        ax.scatter(x, y, color=color, s=500, zorder=3, alpha=0.85, edgecolors='white', linewidths=2)
        ax.annotate(label, (x, y), textcoords='offset points', xytext=(12, 4),
                    fontsize=12, fontweight='bold', color=color)

    ax.set_xlim(coords[:, 0].min() - x_pad, coords[:, 0].max() + x_pad)
    ax.set_ylim(coords[:, 1].min() - y_pad, coords[:, 1].max() + y_pad)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=13, pad=12)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.grid(True, alpha=0.15, linestyle='--')
    plt.tight_layout()
    plt.show()

plot_embedding_map(words, embeddings)

## 🎮 직접 해 보세요!

원하는 단어나 문구 사이의 유사도를 테스트해 보고 싶나요? 아래 셀을 실행하며 실험해 보세요!

In [ ]:
# 원하는 단어나 문구로 해 보세요!
# 비교하고 싶은 것으로 자유롭게 바꿔 보세요
test_words = [
    "machine learning",
    "artificial intelligence", 
    "deep learning",
    "cooking recipes"
]

# 임베딩 생성
test_embeddings = model.encode(test_words)

# 첫 번째 문구를 나머지 모든 문구와 비교
base_embedding = test_embeddings[0].reshape(1, -1)
other_test_embeddings = test_embeddings[1:]
test_similarities = cosine_similarity(base_embedding, other_test_embeddings)[0]

print(f"🧪 '{test_words[0]}'은 다른 문구들과 얼마나 유사할까요?")
print()
for phrase, similarity in zip(test_words[1:], test_similarities):
    print(f"'{test_words[0]}' ↔ '{phrase}': {similarity:.3f}")

print()
print("💡 위의 test_words 리스트를 수정하고 코드를 다시 실행하여 다양한 결과를 실험해 보세요!")

plot_embedding_map(test_words, test_embeddings)

## 🚀 축하합니다!

이제 임베딩을 사용할 수 있게 되어, 텍스트를 검색하는 가장 유연한 방법 중 하나를 익혔습니다.  
실습 안내 문서로 돌아가 이러한 임베딩과 그에 딸린 텍스트 조각을 어디에 저장하고 어떻게 검색하는지 알아보세요.